# Chapter 8 assignment: a delay integro-differential equation

Companion notebook for Chapter 8 (*Integro-Differential Equations*), Exercise "A delay
integro-differential equation".

**Learning goals**

1. Identify the breaking points of a delay equation and their effect on smoothness.
2. Build a classical reference by the method of steps (DOP853 on each delay interval).
3. Implement the *history gate* and an auxiliary-variable PINN (A-PINN) for a delay IDE,
   with the initial condition imposed exactly.
4. Measure the PINN error against the reference and relate it to the breaking points.

Everything runs on a CPU in a few minutes (double precision, one thread).

## 1. Problem and short theory recap

On $[0,1]$ we solve

$$u'(x)=\alpha\,u(x)+\beta\,u(x-\tau)+\lambda\,x\int_0^x t\,u(t-\tau)\,dt,\qquad
u(x)=\phi(x)=1+x\ \text{ for } -\tau\le x\le 0,$$

with $\alpha=1$, $\beta=\tfrac12$, $\lambda=1$, $\tau=0.2$.

* **History moment.** $C(x)=\int_0^x t\,u(t-\tau)\,dt$ satisfies $C'(x)=x\,u(x-\tau)$,
  $C(0)=0$, and $u'=\alpha u+\beta u(x-\tau)+\lambda x C$.
* **Method of steps.** On $[k\tau,(k+1)\tau]$ the delayed values are known (from $\phi$ for
  $k=0$, from the previous step otherwise), so each step is an ODE system for $(u,C)$.
* **Breaking points.** $u'(0^+)=\alpha\phi(0)+\beta\phi(-\tau)\neq\phi'(0^-)$, so $u'$ jumps
  at $x=0$; the jump propagates to $u''$ at $x=\tau$, to $u'''$ at $x=2\tau$, and so on.
* **History gate.** The network represents $u$ on $[0,1]$ only;
  $\tilde u_\theta(\xi)=\phi(\xi)$ for $\xi<0$ and $\tilde u_\theta(\xi)=u_\theta(\xi)$ for
  $\xi\ge 0$. With $u_\theta(x)=\phi(0)+x\,N_\theta(x)$ the condition $u(0)=\phi(0)$ holds
  exactly.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
from scipy.integrate import solve_ivp
from pinnode import (MLP, ConstrainedModel, derivative, to_tensor, train, rel_l2, linf,
                     uniform, volterra_gauss, configure_torch, set_seed)
from pinnode import plotting as P

configure_torch("float64", threads=1)
set_seed(1234)
ALPHA, BETA, LAM, TAU = 1.0, 0.5, 1.0, 0.2
phi = lambda x: 1.0 + x

## 2. Task 1 (analytical): breaking points

Compute $u'(0^+)$ from the equation, and the jump $[u''](\tau)=u''(\tau^+)-u''(\tau^-)$.
*Hint:* differentiate the equation; only the term $\beta\,u'(x-\tau)$ is discontinuous at
$x=\tau$.

In [ ]:
du_right_0 = None      # TODO: u'(0+)
jump_d2u_tau = None    # TODO: u''(tau+) - u''(tau-)

In [ ]:
# check your answer
assert du_right_0 is not None and np.isclose(du_right_0, 1.4), "u'(0+) = alpha*phi(0) + beta*phi(-tau)"
assert jump_d2u_tau is not None and np.isclose(jump_d2u_tau, 0.2), "beta * (u'(0+) - phi'(0-))"
print("Task 1 OK")

## 3. Task 2: method-of-steps reference

Complete the right-hand side of the $(u,C)$ system. `u_delayed(x)` returns $u(x-\tau)$ from
the history or from the dense output of the previous steps.

In [ ]:
def method_of_steps(n_steps=5, rtol=1e-12, atol=1e-12):
    segments = []

    def u_delayed(x):
        s = x - TAU
        if s <= 0.0:
            return phi(s)
        k = min(int(s // TAU), len(segments) - 1)
        return segments[k].sol(s)[0]

    y = np.array([phi(0.0), 0.0])
    for k in range(n_steps):
        def rhs(x, yy):
            ud = u_delayed(x)
            # TODO: return [u'(x), C'(x)]
            raise NotImplementedError
        sol = solve_ivp(rhs, (k * TAU, (k + 1) * TAU), y, method="DOP853",
                        rtol=rtol, atol=atol, dense_output=True)
        segments.append(sol)
        y = sol.y[:, -1]

    def u(x):
        x = np.atleast_1d(x)
        k = np.minimum((x // TAU).astype(int), n_steps - 1)
        return np.array([segments[ki].sol(xi)[0] for ki, xi in zip(k, x)])
    return u

u_ref_fn = method_of_steps()
x_test = np.linspace(0.0, 1.0, 1001)
u_ref = u_ref_fn(x_test)

In [ ]:
# check your answer (values from chapters/ch08_integro_differential/delay_reference.py)
assert abs(u_ref_fn(0.5)[0] - 2.007184431334) < 1e-8
assert abs(u_ref_fn(1.0)[0] - 4.240158013360) < 1e-8
print("Task 2 OK: u(1) =", u_ref_fn(1.0)[0])

## 4. Task 3: history gate and A-PINN

The network has two outputs, $u_\theta=\phi(0)+x\,N_1(x)$ and $C_\theta=x\,N_2(x)$.
Complete

* `u_gate(xi)`: the history gate (use `torch.where`; the network can be evaluated at
  negative arguments, the values are simply not selected there);
* `loss_fn()`: the residuals
  $R_u=u_\theta'-\alpha u_\theta-\beta\tilde u_\theta(x-\tau)-\lambda x C_\theta$ and
  $R_C=C_\theta'-x\,\tilde u_\theta(x-\tau)$, and the loss
  $\overline{R_u^2}+\overline{R_C^2}$ (means over the collocation points).

In [ ]:
torch.manual_seed(1234)
x_col = uniform(100, 0.0, 1.0)
xc = to_tensor(x_col, requires_grad=True)
net = MLP(1, 2, width=32, depth=3, activation="tanh", domain=(0.0, 1.0))
model = ConstrainedModel(net, lambda x, n: torch.cat([phi(0.0) + x * n[:, :1], x * n[:, 1:2]], dim=1))

def u_theta(x):
    return model(x)[:, :1]

def u_gate(xi):
    # TODO: phi(xi) where xi < 0, u_theta(xi) elsewhere
    raise NotImplementedError

def loss_fn():
    out = model(xc)
    d = derivative(out, xc)
    u, C = out[:, :1], out[:, 1:2]
    du, dC = d[:, :1], d[:, 1:2]
    ud = u_gate(xc - TAU)
    # TODO: residuals R_u, R_C and the loss
    raise NotImplementedError
    return loss, {"R_u": L_u, "R_C": L_C}

In [ ]:
hist = train(model.parameters(), loss_fn, adam_iters=4000, lr=1e-3, lr_final=1e-5,
             lbfgs_iters=500, print_every=1000)
with torch.no_grad():
    u_pinn = u_theta(to_tensor(x_test)).numpy().ravel()
err = rel_l2(u_pinn, u_ref)
print(f"relative L2 error: {err:.2e},  max error: {linf(u_pinn, u_ref):.2e}")

In [ ]:
# check your answer
assert err < 1e-4, "expected a relative L2 error below 1e-4; check the gate and the residuals"
print("Task 3 OK")

## 5. Task 4: where is the error largest?

Plot the pointwise error $|u_\theta-u_{\rm ref}|$ on a logarithmic axis and mark the
breaking points $x=\tau$ and $x=2\tau$. Then train the *quadrature* variant (single output,
residual with the Gauss rule `volterra_gauss(x_col, 20)` for
$\int_0^{x_i} t\,\tilde u_\theta(t-\tau)\,dt$) and compare the two errors.

In [ ]:
fig, ax = P.figure(1, 1, width="half", aspect=0.75)
P.plot_error(ax, x_test[1:], u_pinn[1:] - u_ref[1:], label="A-PINN", xlabel="$x$")
for xb in (TAU, 2 * TAU):
    ax.axvline(xb, color=P.COLORS["muted"], lw=0.6)
# TODO: add the quadrature PINN (single output u_theta = phi(0) + x N(x))

**Questions.** (i) Is the error correlated with the breaking points? Why are their effects
small? (ii) Why would it be a mistake to train the network to match $\phi$ on $[-\tau,0]$
as well? (iii) Which variant is cheaper per iteration, and why?